# IT3091 - Member 3 - Weekly wholesale price preprocessing


In [1]:
from pathlib import Path
import os, subprocess
REPO_URL = "https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git"
REPO_PATH = Path("/content/IT3091---Machine-Learning-Assignment")
if not REPO_PATH.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
os.chdir(REPO_PATH)
print("Repository:", REPO_PATH)

Repository: /content/IT3091---Machine-Learning-Assignment


## Build the weekly wholesale price dataset

In [2]:
!pip -q install openpyxl joblib
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
RAW_PATH = REPO_PATH / "Raw Datasets"
PREPROCESSED_PATH = REPO_PATH / "Preprocessed Datasets"
PREPROCESSED_PATH.mkdir(parents=True, exist_ok=True)
matches = list(RAW_PATH.glob("*Weekly*Average*Retail*Prices*.xlsx")) or list(RAW_PATH.glob("*Weekly*Prices*.xlsx"))
if not matches:
    raise FileNotFoundError("Put the weekly wholesale workbook in Raw Datasets first.")
PRICE_FILE = matches[0]
preview = pd.read_excel(PRICE_FILE, header=None, nrows=20)
header_row = next((i for i,row in preview.iterrows() if row.astype(str).str.contains("Market Type", case=False).any()), 0)
raw = pd.read_excel(PRICE_FILE, header=header_row)
raw.columns = [str(c).strip() for c in raw.columns]
market_col = next((c for c in raw.columns if c.lower() == "market type"), None)
if market_col:
    raw = raw[raw[market_col].astype(str).str.lower().str.contains("wholesale", na=False)]
crop_col = next((c for c in raw.columns if c.lower() == "crop"), raw.columns[0])
year_col = next((c for c in raw.columns if c.lower() == "year"), None)
week_cols = [c for c in raw.columns if re.fullmatch(r"W\s*\d{1,2}", str(c), re.I)]
if year_col is None or not week_cols:
    raise ValueError("Year or W1-W52 columns were not found: " + str(list(raw.columns)))
long = raw[[crop_col, year_col] + week_cols].rename(columns={crop_col:"Crop", year_col:"Year"}).melt(id_vars=["Crop","Year"], var_name="Week", value_name="WholesalePrice_RsKg")
long["Crop"] = long["Crop"].astype(str).str.strip()
long["Year"] = pd.to_numeric(long["Year"], errors="coerce")
long["WeekNumber"] = long["Week"].astype(str).str.extract(r"(\d+)")[0].astype(float)
long["WholesalePrice_RsKg"] = pd.to_numeric(long["WholesalePrice_RsKg"].astype(str).str.replace(",","",regex=False), errors="coerce")
long = long.dropna(subset=["Crop","Year","WeekNumber","WholesalePrice_RsKg"]).copy()
long["Year"] = long["Year"].astype(int)
long["WeekNumber"] = long["WeekNumber"].astype(int).clip(1,52)
long["WeekDate"] = pd.to_datetime(long["Year"].astype(str) + "-W" + long["WeekNumber"].astype(str) + "-1", format="%G-W%V-%u", errors="coerce")
weekly = long.groupby(["Crop","WeekDate"], as_index=False)["WholesalePrice_RsKg"].mean().sort_values(["Crop","WeekDate"])
g = weekly.groupby("Crop", group_keys=False)
weekly["PreviousWeekWholesalePrice_RsKg"] = g["WholesalePrice_RsKg"].shift(1)
weekly["NextWeekWholesalePrice_RsKg"] = g["WholesalePrice_RsKg"].shift(-1)
weekly["Year"] = weekly["WeekDate"].dt.year.astype(int)
weekly["WeekNumber"] = weekly["WeekDate"].dt.isocalendar().week.astype(int)
weekly["Month"] = weekly["WeekDate"].dt.month.astype(int)
weekly["Market"] = "Colombo and suburbs"
weekly = weekly[["WeekDate","Crop","Market","Year","WeekNumber","Month","WholesalePrice_RsKg","PreviousWeekWholesalePrice_RsKg","NextWeekWholesalePrice_RsKg"]]
output = PREPROCESSED_PATH / "03_price_preprocessed.csv"
weekly.to_csv(output, index=False)
print("Source:", PRICE_FILE.name)
print("Weekly records:", len(weekly))
print("Saved:", output)
display(weekly.head())

Source: Weekly Average Retail Prices of Vegi - 2012 to 2025.xlsx
Weekly records: 608
Saved: /content/IT3091---Machine-Learning-Assignment/Preprocessed Datasets/03_price_preprocessed.csv


,WeekDate,Crop,Market,Year,WeekNumber,Month,WholesalePrice_RsKg,PreviousWeekWholesalePrice_RsKg,NextWeekWholesalePrice_RsKg
0,2012-01-02,nan,Colombo and suburbs,2012,1,1,64.864583,NaN,54.519167
1,2012-01-09,nan,Colombo and suburbs,2012,2,1,54.519167,64.864583,50.918400
2,2012-01-16,nan,Colombo and suburbs,2012,3,1,50.918400,54.519167,47.436400
3,2012-01-23,nan,Colombo and suburbs,2012,4,1,47.436400,50.918400,39.148400
4,2012-01-30,nan,Colombo and suburbs,2012,5,1,39.148400,47.436400,36.171200


In [ ]:
from getpass import getpass
import json
import os, subprocess
from pathlib import Path
try:
    from google.colab import _message
    current_notebook = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
    notebook_path = REPO_PATH / "03_IT24101090_JAYAKODY_ND_Price_Weekly_Only.ipynb"
    notebook_path.write_text(json.dumps(current_notebook["ipynb"], ensure_ascii=False, indent=1), encoding="utf-8")
except Exception as exc:
    print("Notebook auto-save was skipped:", exc)
github_username = input("GitHub username: ").strip()
github_email = input("GitHub email: ").strip()
github_token = getpass("GitHub Personal Access Token: ").strip()
subprocess.run(["git", "config", "user.name", github_username], cwd=REPO_PATH, check=True)
subprocess.run(["git", "config", "user.email", github_email], cwd=REPO_PATH, check=True)
subprocess.run(["git", "add", "03_IT24101090_JAYAKODY_ND_Price_Weekly_Only.ipynb", "Preprocessed Datasets/03_price_preprocessed.csv"], cwd=REPO_PATH, check=True)
subprocess.run(["git", "commit", "-m", "Prepare weekly wholesale price dataset"], cwd=REPO_PATH, check=False)
askpass = Path("/tmp/git_askpass.sh")
askpass.write_text("#!/bin/sh\ncase \"$1\" in\n  *Username*) echo \"$GITHUB_USERNAME\" ;;\n  *Password*) echo \"$GITHUB_TOKEN\" ;;\nesac\n", encoding="utf-8")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({"GITHUB_USERNAME": github_username, "GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass)})
result = subprocess.run(["git", "push", "origin", "main"], cwd=REPO_PATH, env=env, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Push failed. Check token permissions and repository access.")
print("Member 3 weekly contribution pushed successfully.")